# Mamba-Attention Hybrid Training on Colab

This notebook trains the hybrid Mamba-2 + sparse attention LM on a GPU.

**Features:**
- Uses Google Drive for persistent checkpoints
- Auto-resumes from latest checkpoint on each session
- Saves every 500 steps
- Works within Colab's 4-24hr session limits

**Setup:**
- Free tier: T4 GPU, ~4hr session
- Colab Pro: V100/A100, ~24hr session
- ~10 min initial setup (compile mamba kernel)

In [ ]:
# @title 1. Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/mamba-attention'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/code', exist_ok=True)
print(f'Storage: {DRIVE_DIR}')

In [ ]:
# @title 2. Clone code and install dependencies
# Clone the project (or upload manually)
# Replace with your repo URL or use the code from this session

# Option A: Clone from your repo
# !git clone https://github.com/YOUR_USER/mamba-attention-hybrid.git /content/mamba-attention-hybrid

# Option B: Upload code directly (run this cell after uploading)
# If you don't have a repo, upload the src/ folder manually
# Or copy from Drive if previously uploaded
import shutil
CODE_DIR = '/content/mamba-attention-hybrid'

# If code exists in Drive, copy it
if os.path.exists(f'{DRIVE_DIR}/code/src'):
    shutil.copytree(f'{DRIVE_DIR}/code', CODE_DIR, dirs_exist_ok=True)
    print('Copied code from Drive')
else:
    print('Upload code manually or clone from GitHub')
    print('Then run this cell again')

In [ ]:
# @title 3. Install dependencies (takes ~10 min for mamba kernel)
%cd /content/mamba-attention-hybrid

import torch
print(f'GPU: {torch.cuda.get_device_name()}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Install mamba CUDA kernel + torch + other deps
!pip install --quiet mamba-ssm causal-conv1d
!pip install --quiet transformers datasets safetensors wandb tiktoken

print('Dependencies installed')

In [ ]:
# @title 4. Copy code to Drive for persistence
import shutil
# Save code to Drive so future sessions don't need to re-upload
if os.path.exists(f'{DRIVE_DIR}/code/src'):
    shutil.rmtree(f'{DRIVE_DIR}/code/src')
shutil.copytree('/content/mamba-attention-hybrid/src', f'{DRIVE_DIR}/code/src')
shutil.copytree('/content/mamba-attention-hybrid/configs', f'{DRIVE_DIR}/code/configs')
print('Code saved to Drive')

In [ ]:
# @title 5. Configure training
# @markdown ---
# @markdown **Model size**
model_config = 'configs/350m.yaml' # @param ['configs/350m.yaml', 'configs/760m.yaml', 'configs/1.2b.yaml', 'configs/tiny_cpu.yaml']

# @markdown **Data**
use_data_mix = False # @param {type:'boolean'}
dataset_name = 'c4' # @param ['c4', 'wikitext-103-raw-v1']

# @markdown **Training**
batch_size = 4 # @param {type:'integer'}
grad_accum = 2 # @param {type:'integer'}
max_steps = 50000 # @param {type:'integer'}
save_interval = 500 # @param {type:'integer'}
precision = 'bf16' # @param ['bf16', 'fp32']

# @markdown **Wandb (optional)**
use_wandb = False # @param {type:'boolean'}
wandb_key = '' # @param {type:'string'}

print(f'Config: {model_config}')
print(f'Batch: {batch_size} × {grad_accum} = {batch_size*grad_accum} effective')
print(f'Steps: {max_steps}')

In [ ]:
# @title 6. Find latest checkpoint for resume
import os, glob

CKPT_DIR = f'{DRIVE_DIR}/checkpoints'
resume_path = None

step_dirs = [d for d in os.listdir(CKPT_DIR) if d.startswith('step_')]
if step_dirs:
    steps = [int(d.replace('step_', '')) for d in step_dirs]
    best = step_dirs[steps.index(max(steps))]
    resume_path = os.path.join(CKPT_DIR, best)
    print(f'Resuming from: {resume_path} (step {max(steps)})')
else:
    print('Starting fresh training')

In [ ]:
# @title 7. Start training
%cd /content/mamba-attention-hybrid

import sys
sys.path.insert(0, '/content/mamba-attention-hybrid')
from src.model.config import ModelConfig
from src.model.model import MambaAttentionLM
from src.training.optimizer import configure_optimizer, get_cosine_schedule

# Build args
class Args:
    pass
args = Args()
args.seed = 42
args.batch_size = batch_size
args.grad_accum = grad_accum
args.max_steps = max_steps
args.warmup_steps = 2000
args.lr = 3e-4
args.min_lr = 1e-5
args.weight_decay = 0.1
args.grad_clip = 1.0
args.precision = precision
args.log_interval = 10
args.save_interval = save_interval
args.dataset = dataset_name
args.data_mix = None if not use_data_mix else None
args.save_dir = CKPT_DIR
args.resume = resume_path
args.wandb = use_wandb
args.wandb_project = 'mamba-attention-hybrid'
args.wandb_run = 'colab'
args.no_auto_resume = True

config = ModelConfig.from_yaml(model_config)
device = torch.device('cuda')
torch.manual_seed(args.seed)

model = MambaAttentionLM(config).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} params')

# Handle resume
optimizer = configure_optimizer(model, weight_decay=args.weight_decay, lr=args.lr)
scheduler = get_cosine_schedule(optimizer, args.warmup_steps, args.max_steps, args.min_lr)

step = 0
tokens_seen = 0
if args.resume:
    from src.model.save_load import load_checkpoint
    model, extra = load_checkpoint(args.resume, device=device)
    if 'optimizer_state_dict' in extra:
        optimizer.load_state_dict(extra['optimizer_state_dict'])
    step = extra.get('step', 0)
    tokens_seen = extra.get('tokens_seen', 0)
    for _ in range(step):
        scheduler.step()
    print(f'Resumed at step {step}, tokens {tokens_seen:,}')

if use_wandb and wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    wandb.init(project=args.wandb_project, name=args.wandb_run or None)

# Data loader
from src.training.data import create_dataloader
dataloader = create_dataloader(
    batch_size=args.batch_size,
    max_seq_len=config.max_seq_len,
    dataset_name=args.dataset,
    split='train',
)

# Training loop
import math, time
scaler = torch.amp.GradScaler('cuda', enabled=(args.precision == 'bf16'))
model.train()

print('Training...')
while step < args.max_steps:
    for batch in dataloader:
        if step >= args.max_steps:
            break

        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda', dtype=torch.bfloat16,
                                 enabled=(args.precision == 'bf16')):
            _, loss = model(input_ids, labels)

        scaler.scale(loss).backward()

        if (step + 1) % args.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        tokens_seen += input_ids.numel()

        if step % args.log_interval == 0:
            ppl = math.exp(min(loss.item(), 20))
            print(f'step={step:>5d} | loss={loss.item():.4f} | ppl={ppl:.2f} | '
                  f'lr={scheduler.get_last_lr()[0]:.2e} | tok={tokens_seen:,}')
            if use_wandb and wandb_key:
                wandb.log({'loss': loss.item(), 'perplexity': ppl,
                          'step': step, 'tokens': tokens_seen})

        if step % args.save_interval == 0 and step > 0:
            from src.model.save_load import save_checkpoint
            save_checkpoint(args.save_dir, f'step_{step}', model, config,
                step=step, optimizer_state_dict=optimizer.state_dict(),
                loss=loss.item(), tokens_seen=tokens_seen)

        step += 1

# Final save
from src.model.save_load import save_checkpoint
save_checkpoint(args.save_dir, 'final', model, config,
    step=step, loss=loss.item(), tokens_seen=tokens_seen)
print('Training complete!')

In [ ]:
# @title 8. Generate sample text (eval)
import torch
from src.model.tokenizer import get_tokenizer

tokenizer = get_tokenizer()
prompt = 'The future of AI is'
ids = tokenizer.encode(prompt)
input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

model.eval()
with torch.no_grad():
    for _ in range(50):
        logits, _ = model(input_ids)
        logits = logits[:, -1, :] / 0.8
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

print('Prompt:', prompt)
print('Output:', tokenizer.decode(input_ids[0].tolist()))

## Next Session: Resume Training

When your Colab session expires, just re-run:
1. **Step 1** (mount Drive)
2. **Step 3** (install deps)
3. **Step 5** (configure)
4. **Step 6** (auto-finds latest checkpoint)
5. **Step 7** (resumes training)

Checkpoints persist in Drive forever.